# Customer Service Swarm | Swarm (Decentralized Handoff)

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph_swarm import create_handoff_tool, create_swarm
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# In-memory order database (simulates a real data layer)
ORDER_DB = {
    "12345": {"product": "Wireless Headphones", "status": "delivered", "amount": 79.99, "refunded": False},
    "12346": {"product": "USB-C Cable", "status": "shipped", "amount": 12.99, "refunded": False},
    "12347": {"product": "Laptop Stand", "status": "processing", "amount": 45.00, "refunded": False},
}

In [5]:
# Real tools backed by the in-memory database
@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by ID and return its details."""
    order = ORDER_DB.get(order_id)
    if not order:
        return f"Order {order_id} not found in our system."
    return (
        f"Order #{order_id}:\n"
        f"  Product: {order['product']}\n"
        f"  Status: {order['status']}\n"
        f"  Amount: ${order['amount']:.2f}\n"
        f"  Refunded: {'Yes' if order['refunded'] else 'No'}"
    )

@tool
def process_refund(order_id: str, reason: str) -> str:
    """Process a refund for an order. Updates the order status in the database."""
    order = ORDER_DB.get(order_id)
    if not order:
        return f"Refund failed: Order {order_id} not found."
    if order["refunded"]:
        return f"Refund failed: Order {order_id} was already refunded."
    if order["status"] not in ("delivered", "shipped"):
        return f"Refund failed: Order {order_id} is still '{order['status']}' and cannot be refunded yet."
    order["refunded"] = True
    order["status"] = "refunded"
    return (
        f"Refund of ${order['amount']:.2f} processed successfully for Order #{order_id}.\n"
        f"Reason: {reason}\n"
        f"Product: {order['product']}"
    )

In [6]:
# Create agents with handoff tools
sales_agent = create_agent(
    model=model,
    tools=[
        lookup_order,
        create_handoff_tool(agent_name="support", description="Transfer to support for technical issues"),
        create_handoff_tool(agent_name="billing", description="Transfer to billing for payment/refund issues"),
    ],
    name="sales",
    system_prompt="You are a sales agent. Help with orders and product inquiries. Look up orders when asked.",
)

support_agent = create_agent(
    model=model,
    tools=[
        create_handoff_tool(agent_name="sales", description="Transfer to sales for order inquiries"),
        create_handoff_tool(agent_name="billing", description="Transfer to billing for payment issues"),
    ],
    name="support",
    system_prompt="You are a technical support agent. Help with technical problems.",
)

billing_agent = create_agent(
    model=model,
    tools=[
        lookup_order,
        process_refund,
        create_handoff_tool(agent_name="sales", description="Transfer to sales for order inquiries"),
        create_handoff_tool(agent_name="support", description="Transfer to support for technical issues"),
    ],
    name="billing",
    system_prompt="You are a billing agent. Handle payments, invoices, and refunds. Always look up the order before processing a refund.",
)

In [7]:
# Create the swarm with checkpointer for active agent tracking across turns
workflow = create_swarm(
    agents=[sales_agent, support_agent, billing_agent],
    default_active_agent="sales"
)

In [8]:
checkpointer = InMemorySaver()
app = workflow.compile(checkpointer=checkpointer)

In [9]:
# Plot the app
plot_mermaid(app)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	sales(sales)
	support(support)
	billing(billing)
	__start__ -.-> billing;
	__start__ -.-> sales;
	__start__ -.-> support;
	billing -.-> sales;
	billing -.-> support;
	sales -.-> billing;
	sales -.-> support;
	support -.-> billing;
	support -.-> sales;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [10]:
# thread_id is required by the checkpointer to track conversation state
result = app.invoke(
    {"messages": [{"role": "user", "content": "I need a refund for order #12345, the product was defective"}]},
    config={"configurable": {"thread_id": "session-001"}}
)

In [11]:
print(result["messages"][-1].content)

The refund for your order #12345 has been successfully processed. The amount of $79.99 will be returned to your original payment method. The reason for the refund was noted as the product being defective. If you have any further questions or need additional assistance, feel free to ask!


In [12]:
stream_invoke(app, {"messages": [{"role": "user", "content": "I need a refund for order #12345, the product was defective"}]},
    config={"configurable": {"thread_id": "session-001"}})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

┌─ HUMAN
│ I need a refund for order #12345, the product was defective
└────────────────────────────────────────

┌─ AI (sales)
│ → tool: transfer_to_billing({})
└────────────────────────────────────────

┌─ TOOL (transfer_to_billing)
│ Successfully transferred to billing
└────────────────────────────────────────

┌─ AI (billing)
│ → tool: lookup_order({'order_id': '12345'})
└────────────────────────────────────────

┌─ TOOL (lookup_order)
│ Order #12345:
│   Product: Wireless Headphones
│   Status: delivered
│   Amount: $79.99
│   Refunded: No
└────────────────────────────────────────

┌─ AI (billing)
│ → tool: process_refund({'order_id': '12345', 'reason': 'Product was defective'})
└────────────────────────────────────────

┌─ TOOL (process_refund)
│ Refund of $79.99 processed successfully for Order #

{'messages': [HumanMessage(content='I need a refund for order #12345, the product was defective', additional_kwargs={}, response_metadata={}, id='9c96415e-7a46-4eba-a314-10c17df3e796'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 117, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f000e9ceb4', 'id': 'chatcmpl-DSUVvCvWAP0ql6SKwpd5OBp7Vw5xM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='sales', id='lc_run--019d6ef2-11e7-7261-94d9-97cca0644ee6-0', tool_calls=[{'name': 'transfer_to_billing', 'args': {}, 'id': 'call_9C2Z08Obb8HtWoXqyfNmCLHC', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met